In [11]:
import sys, os
sys.path.append("../src")
import paths                                     # HF_HOME → D
from paths import DATA_DIR

import torch
import torch.nn.functional as F
from huggingface_hub import hf_hub_download
from transformers import AutoModel, AutoTokenizer

from prepare_sft import REPO, iter_chunks, parse_example

device = "cuda"

# 문서 저장소: valid의 Summary 문장 (원본에는 Summary가 100% 있음)
path = hf_hub_download(REPO, "TinyStories-Instruct-valid.txt", repo_type="dataset")
docs = [e for e in (parse_example(c) for c in iter_chunks(path)) if "summary" in e and e["story"]]
summaries = [d["summary"] for d in docs]
print(f"문서 수: {len(summaries):,}")
print("예시:", summaries[0])

# 문장 임베딩 모델 (약 90MB 다운로드, D의 HF 캐시)
EMB = "sentence-transformers/all-MiniLM-L6-v2"
etok = AutoTokenizer.from_pretrained(EMB)
emodel = AutoModel.from_pretrained(EMB).to(device).eval()


@torch.no_grad()
def embed(texts, bs=256):
    out = []
    for i in range(0, len(texts), bs):
        b = etok(texts[i:i + bs], padding=True, truncation=True,
                 max_length=128, return_tensors="pt").to(device)
        h = emodel(**b).last_hidden_state                  # (B, T, 384) 토큰마다 벡터
        m = b["attention_mask"].unsqueeze(-1).float()      # (B, T, 1)   진짜 토큰 1, 패딩 0
        v = (h * m).sum(dim=1) / m.sum(dim=1)              # (B, 384)    패딩 뺀 평균
        out.append(F.normalize(v, dim=-1))                 # (B, 384)    길이 1 → 내적 = 코사인
    return torch.cat(out)                                  # (N, 384)


index = embed(summaries)                                   # (N, 384) 벡터 인덱스
print("index:", index.shape, index.dtype)
print("벡터 길이 확인:", index[:3].norm(dim=-1))
torch.save({"index": index.cpu(), "summaries": summaries},
           os.path.join(DATA_DIR, "rag_index.pt"))

문서 수: 25,026
예시: Tom and Anna are excited to go on a holiday with their parents, and they fly on a big plane to a place with sun and sand.


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 3525.03it/s]


index: torch.Size([25026, 384]) torch.float32
벡터 길이 확인: tensor([1., 1., 1.], device='cuda:0')


In [12]:
def search(query, k=3):
    q = embed([query])                                     # (1, 384)
    scores = (index @ q.T).squeeze(1)                      # (N, 384) @ (384, 1) → (N,) 모든 문서와 코사인
    top = scores.topk(k)
    print(f"\n질문: {query}")
    for s, i in zip(top.values.tolist(), top.indices.tolist()):
        print(f"  {s:.3f}  {summaries[i]}")


search("a dragon who is scared of fire")
search("a girl loses her teddy bear at the park")
search("two friends fight over a red ball and then share")


질문: a dragon who is scared of fire
  0.652  A colorful dragon has a nightmare and is comforted by his mommy, who helps him feel better and they play together until morning.
  0.632  Lily, a scared princess, mistakes fireworks for a dragon outside her window. Her mom explains what they are and helps her overcome her fear.
  0.622  A little boy is scared of a fake hurricane, but a magical golden dragon appears and uses its powers to make the hurricane go away, leaving the boy happy and grateful.

질문: a girl loses her teddy bear at the park
  0.715  A little girl loses her toy in the park but finds it again in the bushes, and has fun playing with her friends.
  0.676  Lily loses her favorite toy bear named Teddy at the park and a kind lady helps her find it.
  0.643  Lily loses her beloved teddy bear at the park and her mom helps her realize the importance of family.

질문: two friends fight over a red ball and then share
  0.798  Two friends quarrel over which ball to play with, but when 

In [13]:
import random
from transformers import AutoModelForCausalLM


def recall_at(doc_vecs, query_vecs, gold, ks=(1, 5, 10)):
    scores = query_vecs @ doc_vecs.T                       # (Q, N) 모든 질문 × 모든 문서
    top = scores.topk(max(ks), dim=1).indices              # (Q, 10) 질문마다 상위 10개 문서 번호
    gold = torch.tensor(gold, device=top.device)[:, None]  # (Q, 1) 질문마다 정답 문서 번호
    return {f"@{k}": f"{(top[:, :k] == gold).any(dim=1).float().mean().item():.1%}" for k in ks}


random.seed(0)
qi = random.sample(range(len(docs)), 1000)                 # 질문으로 쓸 예시 1,000개
queries = [docs[i]["story"] for i in qi]                   # 질문 = 이야기 본문 (앞 128토큰까지만 사용)

# ① 문장 임베딩 모델 (MiniLM)
print("MiniLM   :", recall_at(index, embed(queries), qi))

# ② 우리 GPT (사전학습 모델의 마지막 블록 출력 평균)
gdir = os.path.join(paths.DATA_ROOT, "hf", "gpt30m")
gtok = AutoTokenizer.from_pretrained(gdir)
gtok.pad_token = gtok.eos_token                            # GPT-2는 패딩 토큰이 없어서 EOT로 대신
gpt = AutoModelForCausalLM.from_pretrained(gdir).to(device).eval()


@torch.no_grad()
def embed_gpt(texts, bs=128):
    out = []
    for i in range(0, len(texts), bs):
        b = gtok(texts[i:i + bs], padding=True, truncation=True,
                 max_length=128, return_tensors="pt").to(device)
        h = gpt.transformer(**b).last_hidden_state                  # (B, T, 384) lm_head 없이 블록 출력만
        m = b["attention_mask"].unsqueeze(-1).float()               # (B, T, 1)
        v = (h * m).sum(dim=1) / m.sum(dim=1)                       # (B, 384)
        out.append(F.normalize(v, dim=-1))
    return torch.cat(out)


gpt_index = embed_gpt(summaries)                           # (N, 384)
print("우리 GPT :", recall_at(gpt_index, embed_gpt(queries), qi))
print(f"무작위   : @10 = {10 / len(docs):.2%}")

MiniLM   : {'@1': '78.3%', '@5': '91.2%', '@10': '93.8%'}


Loading weights: 100%|██████████| 76/76 [00:00<00:00, 1535.90it/s]


우리 GPT : {'@1': '43.2%', '@5': '63.9%', '@10': '71.9%'}
무작위   : @10 = 0.04%


In [14]:
import tiktoken
from model import GPT, GPTConfig
from prepare_sft import build_prompt
from sample import generate

enc = tiktoken.get_encoding("gpt2")
ck = torch.load(os.path.join(paths.CKPT_DIR, "sft30m", "best.pt"), map_location="cpu", weights_only=False)
sft = GPT(GPTConfig(**ck["model_config"]))
sft.load_state_dict(ck["model"])
sft.to(device).eval()

THRESH = 0.5


def retrieve(query):
    q = embed([query])                                     # (1, 384)
    s = (index @ q.T).squeeze(1)                           # (N,)
    score, i = s.max(dim=0)
    return summaries[i.item()], score.item()


def write(summary, seed=0):
    prompt = build_prompt({"summary": summary})            # 5단계와 같은 템플릿: "Summary: …\nStory:\n"
    ids = enc.encode(prompt)
    torch.manual_seed(seed)                                # 두 방식에 같은 난수 조건
    out = generate(sft, torch.tensor([ids], device=device), 256 - len(ids),
                   temperature=0.8, top_k=50, eot=enc.eot_token)[0, len(ids):].tolist()
    if enc.eot_token in out:
        out = out[:out.index(enc.eot_token)]
    return enc.decode(out)


for query in ["a dragon who is scared of fire",
              "a girl loses her teddy bear at the park",
              "two friends fight over a red ball and then share"]:
    doc, score = retrieve(query)
    used = doc if score >= THRESH else query
    print(f"\n{'=' * 70}\n질문: {query}")
    print(f"검색 결과 ({score:.3f}{'' if score >= THRESH else ', 기준 미달 → 질문 사용'}): {doc}")
    print(f"\n[RAG 없음]\n{write(query)[:350]}…")
    print(f"\n[RAG 있음]\n{write(used)[:350]}…")


질문: a dragon who is scared of fire
검색 결과 (0.652): A colorful dragon has a nightmare and is comforted by his mommy, who helps him feel better and they play together until morning.

[RAG 없음]
Once upon a time, there was a dragon who lived in a cave. He was very loud and brave. His name was Mr. Crocodile. He was always loud.
One day, Mr. crocodile was walking around when he saw a flame. It was burning his head! He was so scared that he ran away.
Mr. Cro crocodile felt sad. He thought of a plan to make the flame go away. He used his sharp…

[RAG 있음]
Once upon a time, there was a happy dragon. Every night, he would fly around looking for something special. One day, he had a nightmare. In the nightmare, he felt he was all alone. He was scared, so he flew away and cried.
When he came back, he saw that his mommy was smiling. She had been looking for him. The boy felt so happy and hugged her tight.…

질문: a girl loses her teddy bear at the park
검색 결과 (0.715): A little girl loses her toy in the p

In [16]:
import re
import collections

STOP = set("a an the who is are was were of at in on to and then with his her their its "
           "it they he she over for from by be does not want wants like that".split())


def keywords(q, k=3):
    """질문에서 불용어를 뺀 앞쪽 단어 k개 (학습 데이터의 Words가 항상 3개라서 3개)"""
    ws = [w.strip(".,!?").lower() for w in q.split()]
    return [w for w in ws if w and w not in STOP][:k]


def write_fields(fields, seed):
    prompt = build_prompt(fields)                           # 학습 때와 같은 템플릿
    ids = enc.encode(prompt)
    torch.manual_seed(seed)
    out = generate(sft, torch.tensor([ids], device=device), 256 - len(ids),
                   temperature=0.8, top_k=50, eot=enc.eot_token)[0, len(ids):].tolist()
    if enc.eot_token in out:
        out = out[:out.index(enc.eot_token)]
    return enc.decode(out)


def variants(query, doc):
    kw = keywords(query)
    return {
        "① RAG 없음":        {"summary": query},
        "② RAG만":           {"summary": doc},
        "③ RAG + 질문 연결":  {"summary": f"{query}. {doc}"},
        "④ RAG + 키워드":     {"words": ", ".join(kw), "summary": doc},
    }


QUERIES = [
    "a dragon who is scared of fire",
    "a girl loses her teddy bear at the park",
    "two friends fight over a red ball and then share",
    "a cat who wants to fly like a bird",
    "a boy who is afraid of the dark",
    "a little fish lost in a big ocean",
    "a robot who learns to dance",
    "a dog that finds a treasure in the garden",
    "a girl who bakes a cake for her mom",
    "a bunny who does not want to share carrots",
]
SEEDS = [0, 1, 2]

rows = collections.defaultdict(list)                        # 방식 → [(질문 유사도, 문서 유사도, 키워드 포함률)]
examples_out = {}

for query in QUERIES:
    doc, score = retrieve(query)
    kw = keywords(query)
    q_vec, d_vec = embed([query]), embed([doc])            # (1, 384) 각각
    for name, fields in variants(query, doc).items():
        for seed in SEEDS:
            story = write_fields(fields, seed)
            s_vec = embed([story])                         # (1, 384)
            hit = sum(re.search(rf"\b{re.escape(w)}", story.lower()) is not None for w in kw) / len(kw)
            rows[name].append(((s_vec @ q_vec.T).item(), (s_vec @ d_vec.T).item(), hit))
            if query == QUERIES[0] and seed == 0:
                examples_out[name] = story

print(f"{'방식':18s} {'질문 유사도':>10s} {'문서 유사도':>10s} {'키워드 포함':>10s}")
for name, vals in rows.items():
    q_sim, d_sim, hit = (sum(v[i] for v in vals) / len(vals) for i in range(3))
    print(f"{name:18s} {q_sim:10.3f} {d_sim:10.3f} {hit:10.1%}")

print(f"\n===== 예시: {QUERIES[0]} (키워드 {keywords(QUERIES[0])}) =====")
for name, story in examples_out.items():
    print(f"\n[{name}]\n{story[:300]}…")

방식                     질문 유사도     문서 유사도     키워드 포함
① RAG 없음                0.578      0.538      75.6%
② RAG만                  0.489      0.620      57.8%
③ RAG + 질문 연결           0.564      0.603      76.7%
④ RAG + 키워드             0.563      0.642      74.4%

===== 예시: a dragon who is scared of fire (키워드 ['dragon', 'scared', 'fire']) =====

[① RAG 없음]
Once upon a time, there was a dragon who lived in a cave. He was very loud and brave. His name was Mr. Crocodile. He was always loud.
One day, Mr. crocodile was walking around when he saw a flame. It was burning his head! He was so scared that he ran away.
Mr. Cro crocodile felt sad. He thought of a…

[② RAG만]
Once upon a time, there was a happy dragon. Every night, he would fly around looking for something special. One day, he had a nightmare. In the nightmare, he felt he was all alone. He was scared, so he flew away and cried.
When he came back, he saw that his mommy was smiling. She had been looking fo…

[③ RAG + 질문 연결]
Once upon a ti